# 🧭 RCA Summary — recon `demo`

**2 findings** across **1 table pair(s)** · source dialect: `snowflake`

| Verdict | Count | Meaning |
| :-- | --: | :-- |
| 🔧 Migration-induced | 0 | Fix in the migration |
| 📊 Genuine data difference | 1 | Route to the data owner |
| 🔍 Needs review | 0 | Investigate further |
| ✅ Benign / expected | 1 | No action |

## 🔺 Top priorities

| Severity | Location | Verdict | Fix / next step |
| :-- | :-- | :-- | :-- |
| 🟠 Medium (52) | `dim_customer.loyalty_tier` | 📊 | Route to the data owner: confirm whether target enrichment is intended. |

## 📋 Reconciliation overview (per table pair)

| Target table | Schema | ➖ Missing in target | ➕ Extra in target | 🔤 Mismatched columns | Verdicts |
| :-- | :-: | --: | --: | :-- | :-- |
| `dim_customer` | ✅ | · | · | 2 (`loyalty_tier`, `attributes`) | 📊1 ✅1 |

## 📈 Match rates (row & column level)

Reconciliation health per table pair. **Row match %** = source rows that exist in target *and* match on all columns.

| Target table | Source rows | Target rows | ➖ Missing | ➕ Extra | Mismatched rows | ✅ Row match % |
| :-- | --: | --: | --: | --: | --: | --: |
| `dim_customer` | 100 | 100 | 0 | 0 | 100 | **0.00%** |

**Column-level match %** _(columns not listed matched 100%)_:

| Table.Column | Rows | Mismatches | ✅ Match % |
| :-- | --: | --: | --: |
| `dim_customer.loyalty_tier` | 100 | 100 | 0.00% |
| `dim_customer.attributes` | 100 | 100 | 0.00% |


## 🎯 Findings by verdict _(highest impact first)_

## 📊 Genuine data difference — _Route to the data owner_

| Location | Severity | Category | Conf. | ✔ | Root cause |
| :-- | :-- | :-- | :-: | :-: | :-- |
| `dim_customer.loyalty_tier` | 🟠 Medium | 🌊 upstream_drift | 82% | ✓ | Source LOYALTY_TIER is NULL for all rows while target is populated — a genuine data/provenance difference, ... |

## ✅ Benign / expected — _No action_

| Location | Severity | Category | Conf. | ✔ | Root cause |
| :-- | :-- | :-- | :-: | :-: | :-- |
| `dim_customer.attributes` | 🟢 Low | 🧬 semi_structured | 90% | · | VARIANT re-serialized with keys reordered; payloads are semantically equal. |

---
# 📅 Validation (row & column match %, date-range filterable)

Set `start_date` / `end_date` widgets to validate a slice, then re-run.

In [ ]:
# 📅 Date-range validation — set the window (widgets), then re-run these cells.
# Row match % and per-column match % over an optional date range so you can
# validate a slice of the migration (e.g. one month) rather than the whole table.
dbutils.widgets.text("start_date", "2000-01-01")
dbutils.widgets.text("end_date", "2100-01-01")
START, END = dbutils.widgets.get("start_date"), dbutils.widgets.get("end_date")

def _win(date_col):
    return f"WHERE `{date_col}` BETWEEN '{START}' AND '{END}'" if date_col else ""

def validate_rows(src, tgt, keys, date_col=None):
    name = tgt.split(".")[-1]
    if not keys:  # no join key learned — report counts only (edit keys to enable match)
        return spark.sql(f"""
            SELECT '{name}' AS table,
                   (SELECT count(*) FROM {src} {_win(date_col)}) AS source_rows,
                   (SELECT count(*) FROM {tgt} {_win(date_col)}) AS target_rows,
                   CAST(NULL AS BIGINT) AS matched_keys,
                   CAST(NULL AS DOUBLE) AS row_match_pct
        """)
    on = " AND ".join(f"s.`{k}` = t.`{k}`" for k in keys)
    return spark.sql(f"""
        WITH s AS (SELECT * FROM {src} {_win(date_col)}),
             t AS (SELECT * FROM {tgt} {_win(date_col)})
        SELECT '{name}' AS table,
               (SELECT count(*) FROM s) AS source_rows,
               (SELECT count(*) FROM t) AS target_rows,
               (SELECT count(*) FROM s JOIN t ON {on}) AS matched_keys,
               round(100.0 * (SELECT count(*) FROM s JOIN t ON {on}) /
                     nullif((SELECT count(*) FROM s), 0), 2) AS row_match_pct
    """)

def validate_column(src, tgt, keys, col, date_col=None):
    on = " AND ".join(f"s.`{k}` = t.`{k}`" for k in keys) if keys else "TRUE"
    return spark.sql(f"""
        WITH s AS (SELECT * FROM {src} {_win(date_col)}),
             t AS (SELECT * FROM {tgt} {_win(date_col)})
        SELECT '{col}' AS column, count(*) AS compared,
               sum(CASE WHEN s.`{col}` <=> t.`{col}` THEN 1 ELSE 0 END) AS matches,
               round(100.0 * sum(CASE WHEN s.`{col}` <=> t.`{col}` THEN 1 ELSE 0 END) /
                     nullif(count(*), 0), 2) AS match_pct
        FROM s JOIN t ON {on}
    """)


In [ ]:
# Row-level match per table pair (edit date_col via the widgets above):
row_checks = [
    validate_rows("fevm_ps_dr_us_east_2_catalog.mig_source_sim.dim_customer", "fevm_ps_dr_us_east_2_catalog.mig_target.dim_customer", ['customer_id'], None),
]
from functools import reduce
reduce(lambda a, b: a.unionByName(b), row_checks).display()

In [ ]:
# Column-level match % (over the same date window):
col_checks = [
    validate_column("fevm_ps_dr_us_east_2_catalog.mig_source_sim.dim_customer", "fevm_ps_dr_us_east_2_catalog.mig_target.dim_customer", ['customer_id'], "loyalty_tier", None),
    validate_column("fevm_ps_dr_us_east_2_catalog.mig_source_sim.dim_customer", "fevm_ps_dr_us_east_2_catalog.mig_target.dim_customer", ['customer_id'], "attributes", None),
]
reduce(lambda a, b: a.unionByName(b), col_checks).display()

---
# 🔬 Findings & evidence

Grouped by table pair (as Lakebridge reports), then schema → row-level → column-level. Each finding shows the concluded verdict and the query that confirms it. Re-run any cell to drill deeper.

## 📦 `fevm_ps_dr_us_east_2_catalog.mig_target.dim_customer`  
_📊1 ✅1  ·  2 finding(s)_

### ✅ `dim_customer.attributes` — Benign / expected

- **Category**: 🧬 semi_structured  ·  **Confidence**: 90%  ·  **Owner**: —
- **Signal**: **column-level** mismatch — `attributes` differs on 100 of 100 rows
- **Root cause**: VARIANT re-serialized with keys reordered; payloads are semantically equal.
- **Fix**: No action (representation-only).
- **Inputs used**: 📊 recon data

Sample differences:
  - `{'customer_id': 1}` source='{"segment":"A","channel":"web"}' → target='{"channel":"web","segment":"A"}'

In [ ]:
# Re-run to confirm / drill deeper for dim_customer.attributes
# spark.sql("SELECT * FROM fevm_ps_dr_us_east_2_catalog.mig_target.dim_customer LIMIT 20").display()

### 📊 `dim_customer.loyalty_tier` — Genuine data difference

- **Category**: 🌊 upstream_drift  ·  **Confidence**: 82%  ·  **Owner**: data owner
- **Signal**: **column-level** mismatch — `loyalty_tier` differs on 100 of 100 rows
- **Root cause**: Source LOYALTY_TIER is NULL for all rows while target is populated — a genuine data/provenance difference, not a migration bug.
- **Fix**: Route to the data owner: confirm whether target enrichment is intended.
- **Inputs used**: 📊 recon data

In [ ]:
# Re-run to confirm / drill deeper for dim_customer.loyalty_tier
# spark.sql("SELECT * FROM fevm_ps_dr_us_east_2_catalog.mig_target.dim_customer LIMIT 20").display()

---
# 🧾 Conclusion & recommended actions

Analyzed **2 findings**. Every verdict below is backed by a query executed in this notebook (see the cell under each finding).

## 🔧 Fix in the migration — 0 (owner: migration engineer)
- _None._

## 📊 Route to the data owner — 1 (not migration bugs)
- `dim_customer.loyalty_tier` — Route to the data owner: confirm whether target enrichment is intended.

## ✅ Benign / expected — 1
- 1 finding(s) are representation-only or within tolerance; no action.

> If re-running a cell changes an output, update that finding's verdict above and regenerate this report so the conclusion always matches the evidence.